In [1]:
import pandas as pd
import numpy as np 
import joblib

In [2]:
df = pd.read_excel(r"../Excel/mutual_funds_.xlsx")
df.head()

,scheme_name,min_sip,min_lumpsum,expense_ratio,fund_size_cr,fund_age_yr,fund_manager,sortino_ratio,alpha,standard_deviation,...,sub_category,returns_1yr,returns_3yr,returns_5yr,risk-adjusted return score,cost efficiency score,consistency score,fund stability,composite_score,rank
0,Quant Infrastructure Fund - Regular Plan IDCW,1000,5000,0.586682,888.937047,10,Vasav Sahgal,3.726774,27.810968,24.259553,...,Sectoral / Thematic Mutual Funds,3.958764,63.090821,22.539075,2.600659,1.704501,29.862887,36.642763,0.704149,1.0
1,Quant Infrastructure Fund - Direct Plan Growth,1000,5000,0.629340,851.092153,11,Vasav Sahgal,3.408510,28.792777,24.292619,...,Sectoral / Thematic Mutual Funds,8.376421,60.112592,22.557321,2.474521,1.588966,30.348778,35.03501,0.701785,2.0
2,Quant Infrastructure Fund - Regular Plan Growth,1000,5000,0.577551,731.522839,10,Vasav Sahgal,3.767372,27.471415,25.389421,...,Sectoral / Thematic Mutual Funds,7.311768,62.288565,20.664012,2.453328,1.731449,30.088115,28.812112,0.697625,3.0
3,Quant Infrastructure Fund - Regular Plan IDCW,1000,5000,0.591545,768.126541,9,Vasav Sahgal,3.516540,26.538881,22.417272,...,Sectoral / Thematic Mutual Funds,5.956095,62.014558,22.404504,2.766374,1.690489,30.125053,34.264943,0.697512,4.0
4,Quant Infrastructure Fund - Regular Plan Growth,1000,5000,0.624836,879.865799,9,Vasav Sahgal,3.456080,28.775877,21.597198,...,Sectoral / Thematic Mutual Funds,8.193366,60.586547,21.827366,2.805297,1.600421,30.202426,40.739812,0.697401,5.0


In [3]:
df = df[
    [
        "scheme_name",
        "returns_5yr",
        "sharpe",
        "standard_deviation",
        "risk_bucket"
    ]
]

In [4]:
df = df.dropna().reset_index(drop=True)

In [6]:
z_cols = ["returns_5yr", "sharpe", "standard_deviation"]

df[z_cols] = (
    df[z_cols]
    .replace("-", pd.NA)
    .apply(pd.to_numeric, errors="coerce")
    .astype(float)
)

In [7]:
df[z_cols] = df[z_cols].fillna(df[z_cols].median())

In [8]:
from scipy.stats import zscore

df["z_returns_5yr"] = zscore(df["returns_5yr"])
df["z_sharpe"] = zscore(df["sharpe"])
df["z_std_dev"] = -zscore(df["standard_deviation"])  # lower risk = better

In [9]:
df["raw_target_score"] = (
    0.5 * df["z_returns_5yr"] +
    0.3 * df["z_sharpe"] +
    0.2 * df["z_std_dev"]
)


In [10]:
from sklearn.preprocessing import StandardScaler

df["target_score_z"] = StandardScaler().fit_transform(
    df[["raw_target_score"]]
).ravel()


In [11]:
FEATURES = [
    "z_returns_5yr",
    "z_sharpe",
    "z_std_dev"
]


In [13]:
from xgboost import XGBRegressor

X = df[FEATURES]
y = df["target_score_z"]

model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model.fit(X, y)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [14]:
RISK_MAP = {
    "Low": ["Low Risk", "Moderately Low"],
    "Moderate": ["Moderate", "Moderately Low"],
    "High": ["High Risk", "Moderate"]
}


In [15]:
def filter_by_risk(df, user_risk):
    return df[df["risk_bucket"].isin(RISK_MAP[user_risk])]


In [16]:
def recommend_top_funds(df, user_risk, top_n=5):
    filtered = filter_by_risk(df, user_risk)

    scores = model.predict(filtered[FEATURES])
    filtered = filtered.copy()
    filtered["final_score"] = scores

    return (
        filtered
        .sort_values("final_score", ascending=False)
        .head(top_n)
        [["scheme_name", "final_score", "returns_5yr", "risk_bucket"]]
    )


In [17]:
top_funds = recommend_top_funds(
    df,
    user_risk="Moderate",
    top_n=5
)

top_funds


,scheme_name,final_score,returns_5yr,risk_bucket
469,Kotak Multi Asset Allocator FoF – Dynamic – Di...,0.453642,16.275858,Moderate
452,Kotak Multi Asset Allocator FoF – Dynamic – Di...,0.453642,16.488664,Moderate
423,Kotak Multi Asset Allocator FoF – Dynamic – Di...,0.453642,16.403755,Moderate
594,Kotak Multi Asset Allocator FoF – Dynamic – Di...,0.453642,16.420270,Moderate
527,Kotak Multi Asset Allocator FoF – Dynamic – Di...,0.453642,16.384110,Moderate


In [18]:
def project_sip(monthly_amount, annual_return, years):
    r = annual_return / 100 / 12
    n = years * 12
    return monthly_amount * ((1 + r)**n - 1) / r * (1 + r)


In [19]:
def attach_projection(df, investment_type, amount, years):
    df = df.copy()

    if investment_type == "SIP":
        r = df["returns_5yr"] / 100 / 12
        n = years * 12
        df["projected_value"] = amount * (((1 + r) ** n - 1) / r) * (1 + r)

    elif investment_type == "LUMPSUM":
        r = df["returns_5yr"] / 100
        df["projected_value"] = amount * ((1 + r) ** years)

    df["investment_type"] = investment_type
    df["investment_years"] = years

    return df


In [20]:
final_output = attach_projection(
    top_funds,
    investment_type="SIP",
    amount=5000,
    years=10
)

final_output


,scheme_name,final_score,returns_5yr,risk_bucket,projected_value,investment_type,investment_years
469,Kotak Multi Asset Allocator FoF – Dynamic – Di...,0.453642,16.275858,Moderate,1.508093e+06,SIP,10
452,Kotak Multi Asset Allocator FoF – Dynamic – Di...,0.453642,16.488664,Moderate,1.528305e+06,SIP,10
423,Kotak Multi Asset Allocator FoF – Dynamic – Di...,0.453642,16.403755,Moderate,1.520203e+06,SIP,10
594,Kotak Multi Asset Allocator FoF – Dynamic – Di...,0.453642,16.420270,Moderate,1.521774e+06,SIP,10
527,Kotak Multi Asset Allocator FoF – Dynamic – Di...,0.453642,16.384110,Moderate,1.518335e+06,SIP,10


In [21]:
RETURN_BANDS = {
    "Low": (0, 10),
    "Moderate": (10, 15),
    "High": (15, 25)
}


In [22]:
def filter_by_return_band(df, band):
    low, high = RETURN_BANDS[band]
    return df[(df["returns_5yr"] >= low) & (df["returns_5yr"] <= high)]


In [23]:
def recommend_top_funds(df, user_risk, return_band, top_n=5):
    filtered = filter_by_risk(df, user_risk)
    filtered = filter_by_return_band(filtered, return_band)

    scores = model.predict(filtered[FEATURES])
    filtered = filtered.copy()
    filtered["final_score"] = scores

    return (
        filtered
        .sort_values("final_score", ascending=False)
        .head(top_n)
        [["scheme_name", "final_score", "returns_5yr", "risk_bucket"]]
    )


In [24]:
top_funds = recommend_top_funds(
    df,
    user_risk="High",
    return_band="High",
    top_n=5
)

top_funds


,scheme_name,final_score,returns_5yr,risk_bucket
3,Quant Infrastructure Fund - Regular Plan IDCW,1.746138,22.404504,High Risk
38,Quant Infrastructure Fund - Regular Plan IDCW,1.746138,22.359804,High Risk
113,Quant Infrastructure Fund - Regular Plan Growth,1.746138,22.322893,High Risk
15,Quant Infrastructure Fund - Regular Plan Growth,1.746138,22.343586,High Risk
17,Quant Infrastructure Fund - Regular Plan Growth,1.746138,22.412402,High Risk


In [25]:
def confidence_score(row):
    score = (
        0.5 * abs(row["z_returns_5yr"]) +
        0.3 * abs(row["z_sharpe"]) +
        0.2 * abs(row["z_std_dev"])
    )
    return min(round(score * 20, 2), 100)

In [26]:
top_funds = top_funds.merge(
    df[["scheme_name", "z_returns_5yr", "z_sharpe", "z_std_dev"]],
    on="scheme_name",
    how="left"
)

top_funds["confidence_%"] = top_funds.apply(confidence_score, axis=1)

top_funds


,scheme_name,final_score,returns_5yr,risk_bucket,z_returns_5yr,z_sharpe,z_std_dev,confidence_%
0,Quant Infrastructure Fund - Regular Plan IDCW,1.746138,22.404504,High Risk,1.612889,1.804468,-1.759665,33.99
1,Quant Infrastructure Fund - Regular Plan IDCW,1.746138,22.404504,High Risk,1.578052,1.804468,-1.414724,32.27
2,Quant Infrastructure Fund - Regular Plan IDCW,1.746138,22.404504,High Risk,1.016421,1.804468,-1.259405,26.03
3,Quant Infrastructure Fund - Regular Plan IDCW,1.746138,22.404504,High Risk,1.114602,1.804468,-1.455227,27.79
4,Quant Infrastructure Fund - Regular Plan IDCW,1.746138,22.404504,High Risk,1.149870,1.804468,-1.189433,27.08
...,...,...,...,...,...,...,...,...
140,Quant Infrastructure Fund - Regular Plan Growth,1.746138,22.412402,High Risk,1.101009,1.804468,-1.430455,27.56
141,Quant Infrastructure Fund - Regular Plan Growth,1.746138,22.412402,High Risk,1.057785,1.804468,-1.383422,26.94
142,Quant Infrastructure Fund - Regular Plan Growth,1.746138,22.412402,High Risk,0.865467,1.804468,-1.875148,26.98
143,Quant Infrastructure Fund - Regular Plan Growth,1.746138,22.412402,High Risk,0.904209,1.804468,-1.411162,25.51


In [27]:
joblib.dump(model, "xgboost_fund_ranker.pkl")

['xgboost_fund_ranker.pkl']